# NB_repush - Recover a result dataset from a notebook output

The NB0-NB4 pushes silently failed: `kaggle datasets create` exits 0 while
printing `already in use`, so the version-bump never ran and each dataset kept
only the 183 B dry-run marker. The real results are safe in each notebook's
saved output (`/kaggle/working`). This notebook mounts one such output and pushes
it to the correct dataset with the corrected create-or-version logic. No retrain.

**How to use (once per dataset):**
1. Add Input -> your NB0/NB1/NB2/NB3/NB4 **notebook output** (Notebooks tab).
2. Set the three constants in section 1 for that output.
3. Save Version - Save and Run All (Commit). Takes ~1-3 min.
4. Repeat for the next output.

| SRC_MARKER | TARGET_SLUG | TARGET_TITLE |
| --- | --- | --- |
| dataset_characterisation.csv | ausdm-v3-data | AusDM v3 shared data |
| nb1_state.json | ausdm-v3-container-supervised | AusDM v3 container supervised |
| nb2_state.json | ausdm-v3-container-openvocab | AusDM v3 container open-vocabulary |
| nb3_state.json | ausdm-v3-icdar | AusDM v3 ICDAR 2015 |
| nb4_state.json | ausdm-v3-coco | AusDM v3 COCO reference |


## 1. Constants - edit for the mounted output

In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

# ---- edit these three per run ----
SRC_MARKER   = 'nb1_state.json'                    # a file unique to the output you mounted
TARGET_SLUG  = 'ausdm-v3-container-supervised'
TARGET_TITLE = 'AusDM v3 container supervised'
# ----------------------------------

KAGGLE_USERNAME_OVERRIDE = ''   # set if auto-detect returns 'unknown'
print('marker :', SRC_MARKER)
print('target :', TARGET_SLUG)

## 2. Locate the results folder inside the mounted output

In [ ]:
# _find_marker_dir - the folder that holds SRC_MARKER (the pushed results root)
def _find_marker_dir():
    hits = list(KAGGLE_INPUT.rglob(SRC_MARKER))
    assert hits, (f'{SRC_MARKER} not found under /kaggle/input. Add the right notebook '
                  f'output. Present: {[p.name for p in KAGGLE_INPUT.iterdir()] if KAGGLE_INPUT.exists() else "nothing"}')
    return hits[0].parent

SRC_DIR = _find_marker_dir()
print('results root:', SRC_DIR)
files = sorted(p.name for p in SRC_DIR.iterdir())
print('contents    :', files)
# quick size sanity so we know real data (not just a marker) is present
total_mb = sum(p.stat().st_size for p in SRC_DIR.rglob('*') if p.is_file()) / 1e6
print(f'total size  : {total_mb:.1f} MB')

## 3. Kaggle username

In [ ]:
# _kaggle_user - same detection used across the pipeline
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE:
        return KAGGLE_USERNAME_OVERRIDE
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        u = api.config_values.get('username') or api.get_config_value('username')
        if u: return u
    except Exception:
        pass
    u = os.environ.get('KAGGLE_USERNAME')
    if u: return u
    kj = Path.home()/'.kaggle'/'kaggle.json'
    if kj.exists():
        try: return json.loads(kj.read_text())['username']
        except Exception: pass
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.rglob('*'):
            parts = p.parts
            if 'datasets' in parts:
                i = parts.index('datasets')
                if i+1 < len(parts): return parts[i+1]
    return 'unknown'

KAGGLE_USER = _kaggle_user()
print('Kaggle username:', KAGGLE_USER)
assert KAGGLE_USER != 'unknown', 'Set KAGGLE_USERNAME_OVERRIDE in section 1 and re-run.'

## 4. Stage and push (corrected create-or-version)

In [ ]:
PUSH_DIR = KAGGLE_WORKING / 'push_stage'
if PUSH_DIR.exists(): shutil.rmtree(PUSH_DIR)
shutil.copytree(SRC_DIR, PUSH_DIR)
# never push kaggle's own metadata from the source, then write a fresh one
for junk in PUSH_DIR.glob('dataset-metadata.json'):
    junk.unlink()
(PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(
    {'title': TARGET_TITLE, 'id': f'{KAGGLE_USER}/{TARGET_SLUG}',
     'licenses': [{'name': 'CC-BY-SA-4.0'}]}, indent=2))

# _push - success judged from output text, not the return code (the original bug)
def _push(message):
    def _run(sub):
        r = subprocess.run(['kaggle','datasets'] + sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if 'invalid owner id' in out.lower():
        print('CREATE:', out.strip()[-200:]); return False
    if 'is being created' in out.lower() or 'successfully' in out.lower():
        print('CREATE ok'); return True
    vout = _run(['version','-p',str(PUSH_DIR),'-m',message,'-r','zip'])
    print('VERSION:', vout.strip()[-260:])
    return ('is being created' in vout.lower()) or ('successfully' in vout.lower())

ok = _push(f'repush {TARGET_SLUG}')
print('\nrepush:', 'OK' if ok else 'FAILED', '->', f'{KAGGLE_USER}/{TARGET_SLUG}')
if ok:
    print('Verify on the dataset page that size is > a few MB, not 183 B.')

**Note.** After all five datasets show real content, NB5 can mount them as
originally intended. Alternatively, skip this notebook entirely and point NB5 at
the NB0-NB4 notebook outputs directly, since NB5 finds files by name across any
mount.
